In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:


# Paths
RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')

# IMF PortWatch
daily_ports  = pd.read_csv(RAW / 'daily_ports.csv')
disruptions  = pd.read_csv(RAW / 'disruptions.csv')

# UNCTAD — load all years and concatenate
time_in_port = pd.read_csv(RAW / 'port_calls.csv')

port_calls = pd.read_csv(RAW / 'port_calls_arr.csv')

print("daily_ports:  ", daily_ports.shape)
print("disruptions:  ", disruptions.shape)
print("time_in_port: ", time_in_port.shape)
print("port_calls:   ", port_calls.shape)

daily_ports:   (5149090, 30)
disruptions:   (127, 21)
time_in_port:  (1197, 29)
port_calls:    (4653, 8)


In [3]:
# Parse date
daily_ports['date'] = pd.to_datetime(daily_ports['date'])

# Filter to container-relevant ports only
PORTS_OF_INTEREST = ['Singapore', 'Shanghai', 'Rotterdam']
ports_filtered = daily_ports[
    daily_ports['portname'].isin(PORTS_OF_INTEREST)
].copy()

In [4]:
ports_filtered

,date,year,month,day,portid,portname,country,ISO3,portcalls_container,portcalls_dry_bulk,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
350050,2019-04-21,2019,4,21,port1114,Rotterdam,The Netherlands,NLD,18,3,...,415244,882442,90621,0,7523,404,81268,98549,179818,350051
350051,2019-04-22,2019,4,22,port1114,Rotterdam,The Netherlands,NLD,24,5,...,305776,745320,198458,33513,4824,1928,134869,238724,373593,350052
350052,2019-04-23,2019,4,23,port1114,Rotterdam,The Netherlands,NLD,22,4,...,417718,653884,135142,1210,24849,1717,215983,162920,378903,350053
350053,2019-04-24,2019,4,24,port1114,Rotterdam,The Netherlands,NLD,15,2,...,240838,704158,71285,49253,10095,6686,131352,137320,268673,350054
350054,2019-04-25,2019,4,25,port1114,Rotterdam,The Netherlands,NLD,27,4,...,331184,798558,252043,7146,3455,3236,87993,265881,353875,350055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3237369,2026-01-21,2026,1,21,port1201,Singapore,Singapore,SGP,41,3,...,848262,1392019,765366,3058,5905,2238,543727,776568,1320295,3237370
3237370,2026-01-22,2026,1,22,port1201,Singapore,Singapore,SGP,43,3,...,762208,1444728,1001641,0,17615,1557,444813,1020815,1465628,3237371
3237371,2026-01-23,2026,1,23,port1201,Singapore,Singapore,SGP,43,0,...,1065156,1857376,1041960,0,4176,2703,274888,1048840,1323729,3237372
3237372,2026-01-24,2026,1,24,port1201,Singapore,Singapore,SGP,38,3,...,848557,1538028,617541,0,5711,5253,315828,628506,944335,3237373


In [5]:


# Check nulls and zero-call days
print(ports_filtered.isnull().sum())
print(ports_filtered[ports_filtered['portcalls_container'] == 0].shape[0],
      "zero-call days")

# Drop rows where ALL call columns are zero (port inactive / no data)
call_cols = [c for c in ports_filtered.columns if c.startswith('portcalls')]
ports_filtered = ports_filtered[
    ports_filtered[call_cols].sum(axis=1) > 0
].copy()



date                       0
year                       0
month                      0
day                        0
portid                     0
portname                   0
country                    0
ISO3                       0
portcalls_container        0
portcalls_dry_bulk         0
portcalls_general_cargo    0
portcalls_roro             0
portcalls_tanker           0
portcalls_cargo            0
portcalls                  0
import_container           0
import_dry_bulk            0
import_general_cargo       0
import_roro                0
import_tanker              0
import_cargo               0
import                     0
export_container           0
export_dry_bulk            0
export_general_cargo       0
export_roro                0
export_tanker              0
export_cargo               0
export                     0
ObjectId                   0
dtype: int64
0 zero-call days


In [6]:
ports_filtered

,date,year,month,day,portid,portname,country,ISO3,portcalls_container,portcalls_dry_bulk,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
350050,2019-04-21,2019,4,21,port1114,Rotterdam,The Netherlands,NLD,18,3,...,415244,882442,90621,0,7523,404,81268,98549,179818,350051
350051,2019-04-22,2019,4,22,port1114,Rotterdam,The Netherlands,NLD,24,5,...,305776,745320,198458,33513,4824,1928,134869,238724,373593,350052
350052,2019-04-23,2019,4,23,port1114,Rotterdam,The Netherlands,NLD,22,4,...,417718,653884,135142,1210,24849,1717,215983,162920,378903,350053
350053,2019-04-24,2019,4,24,port1114,Rotterdam,The Netherlands,NLD,15,2,...,240838,704158,71285,49253,10095,6686,131352,137320,268673,350054
350054,2019-04-25,2019,4,25,port1114,Rotterdam,The Netherlands,NLD,27,4,...,331184,798558,252043,7146,3455,3236,87993,265881,353875,350055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3237369,2026-01-21,2026,1,21,port1201,Singapore,Singapore,SGP,41,3,...,848262,1392019,765366,3058,5905,2238,543727,776568,1320295,3237370
3237370,2026-01-22,2026,1,22,port1201,Singapore,Singapore,SGP,43,3,...,762208,1444728,1001641,0,17615,1557,444813,1020815,1465628,3237371
3237371,2026-01-23,2026,1,23,port1201,Singapore,Singapore,SGP,43,0,...,1065156,1857376,1041960,0,4176,2703,274888,1048840,1323729,3237372
3237372,2026-01-24,2026,1,24,port1201,Singapore,Singapore,SGP,38,3,...,848557,1538028,617541,0,5711,5253,315828,628506,944335,3237373


In [ ]:
# Parse date
daily_ports['date'] = pd.to_datetime(daily_ports['date'])

# Filter to container-relevant ports only
PORTS_OF_INTEREST = ['Singapore', 'Shanghai', 'Rotterdam']
ports_filtered = daily_ports[
    daily_ports['portname'].isin(PORTS_OF_INTEREST)
].copy()

# Check nulls and zero-call days
print(ports_filtered.isnull().sum())
print(ports_filtered[ports_filtered['portcalls_container'] == 0].shape[0],
      "zero-call days")

# Drop rows where ALL call columns are zero (port inactive / no data)
call_cols = [c for c in ports_filtered.columns if c.startswith('portcalls')]
ports_filtered = ports_filtered[
    ports_filtered[call_cols].sum(axis=1) > 0
].copy()



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')

daily_ports  = pd.read_csv(RAW / 'daily_ports.csv')
disruptions  = pd.read_csv(RAW / 'disruptions.csv')
time_in_port = pd.read_csv(RAW / 'port_calls.csv')
port_calls   = pd.read_csv(RAW / 'port_calls_arr.csv')

# ── 1. DAILY PORTS ──────────────────────────────────────────────────────
print("=" * 60)
print("DAILY PORTS")
print("=" * 60)
print("Shape:", daily_ports.shape)
print("\nDate range:")
print(daily_ports['date'].min(), "→", daily_ports['date'].max())
print("\nAll unique port names:")
print(sorted(daily_ports['portname'].unique().tolist()))
print("\nShanghai variants:")
print(daily_ports[daily_ports['portname'].str.contains('Shanghai', case=False, na=False)]['portname'].unique())
print("\nSingapore variants:")
print(daily_ports[daily_ports['portname'].str.contains('Singapore', case=False, na=False)]['portname'].unique())
print("\nRotterdam variants:")
print(daily_ports[daily_ports['portname'].str.contains('Rotterdam', case=False, na=False)]['portname'].unique())
print("\nPortcalls_container stats for Shanghai Yangshan:")
yangshan = daily_ports[daily_ports['portname'] == 'Shanghai (Yangshan)']
print(yangshan['portcalls_container'].describe())
print("\nNull counts:")
print(daily_ports.isnull().sum())
print("\nSample rows:")
print(daily_ports[daily_ports['portname'].isin(['Shanghai (Yangshan)', 'Singapore', 'Rotterdam'])].head(10))

# ── 2. DISRUPTIONS ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("DISRUPTIONS")
print("=" * 60)
print("Shape:", disruptions.shape)
print("\nColumns:", disruptions.columns.tolist())
print("\nEvent types:")
print(disruptions['eventtype'].value_counts())
print("\nAlert levels:")
print(disruptions['alertlevel'].value_counts())
print("\nDate range (fromdate):")
print(disruptions['fromdate'].min(), "→", disruptions['fromdate'].max())
print("\nSample affectedports values:")
print(disruptions['affectedports'].dropna().head(10).tolist())
print("\nNull counts:")
print(disruptions.isnull().sum())

# ── 3. TIME IN PORT (UNCTAD) ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("TIME IN PORT — UNCTAD")
print("=" * 60)
print("Shape:", time_in_port.shape)
print("\nColumns:", time_in_port.columns.tolist())
print("\nAll CommercialMarket values:")
print(time_in_port[['CommercialMarket','CommercialMarket Label']].drop_duplicates().to_string())
print("\nAll Economy codes for Singapore, China, Netherlands:")
for name in ['Singapore', 'China', 'Netherlands']:
    rows = time_in_port[time_in_port['Economy Label'].str.contains(name, case=False, na=False)]
    print(f"\n  {name}:")
    print(rows[['Year','Economy','Economy Label','CommercialMarket','CommercialMarket Label','Median time in port (days)']].to_string())
print("\nContainer ships (CommercialMarket==03) for our 3 countries:")
container = time_in_port[time_in_port['CommercialMarket'].astype(str).str.strip() == '03']
our_economies = [156, 528, 702]
print(container[container['Economy'].isin(our_economies)][
    ['Year','Economy Label','Median time in port (days)',
     'Average size (GT) of vessels']
].to_string())
print("\nNull counts on key column:")
print(time_in_port['Median time in port (days)'].isnull().sum(), "nulls in median_time_in_port")

# ── 4. PORT CALLS (UNCTAD) ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("PORT CALLS — UNCTAD")
print("=" * 60)
print("Shape:", port_calls.shape)
print("\nColumns:", port_calls.columns.tolist())
print("\nContainer ships for our 3 countries:")
container_calls = port_calls[port_calls['CommercialMarket'].astype(str).str.strip() == '03']
print(container_calls[container_calls['Economy'].isin(our_economies)][
    ['Year','Economy Label','Number of port calls']
].to_string())
print("\nNull counts:")
print(port_calls.isnull().sum())

# ── 5. PORTID MAPPING ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PORTID MAPPING")
print("=" * 60)
port_id_map = daily_ports[
    daily_ports['portname'].isin([
        'Shanghai (Yangshan)', 'Singapore', 'Rotterdam'
    ])
][['portid','portname','country','ISO3']].drop_duplicates()
print(port_id_map.to_string())

DAILY PORTS
Shape: (5149090, 30)

Date range:
2019/01/01 → 2026/02/06

All unique port names:
['Aabenraa', 'Aalborg', 'Aamchit', 'Aarhus', 'Abashiri', 'Abbot Point', 'Aberdeen', 'Abidjan', 'Abu Dhabi', 'Aburatsu', 'Acajutla', 'Acapulco', 'Aden', 'Agadir', 'Agioi Oil Terminal', 'Ahus', 'Aiyion', 'Ajaccio', 'Ajman', 'Akita', 'Ako', 'Aktau Port', 'Akureyri', 'Al Adabiyah', 'Al Basrah', 'Al Fujayrah', 'Al Hamriyah LPG Terminal', 'Al Hoceima', 'Al Ruwais', 'Albany', 'Alesund', 'Alexandria', 'Alexandroupoli', 'Algeciras', 'Alger', 'Aliaga', 'Alicante', 'Almeria', 'Almirante', 'Alotau', 'Altamira', 'Alvika', 'Amagasaki-Nishinomiya-Ashiya', 'Amamapare', 'Ambarli', 'Amirabad Port', 'Amlan', 'Ampenan', 'Amsterdam', 'Amuay (Bahia De Amuay)', 'Amurang', 'Anacortes', 'Anchorage (Alaska)', 'Ancona', 'Andoany', 'Angola - Offshore Oil Terminal 1', 'Angola - Offshore Oil Terminal 2', 'Angola - Offshore Oil Terminal 3', 'Angola - Offshore Oil Terminal 4', 'Angola - Offshore Oil Terminal 5', 'Angola - Of